# Exploratory Data Analysis: Market Observations & Returns

**Scope:**
- Audit missing value profiles across raw FRED observations.
- Visualize price levels and yield trajectories.
- Analyze daily return distributions, rolling volatility, and cross-asset correlations.

*Note: Data cleaning and return transformations are loaded directly from `data/processed/`.*

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Visualization setup
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

--- 
## 1. Raw Data Quality & Missing Value Audit

In [ ]:
# Load quality summary produced by download_market_data.py
quality_path = RAW_DIR / 'quality_summary.json'

if quality_path.exists():
    with open(quality_path, 'r') as f:
        quality_data = json.load(f)
    
    df_quality = pd.DataFrame.from_dict(quality_data['metrics'], orient='index')
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Total vs Missing Observations
    df_quality[['total_row_count', 'missing_observations_count']].plot(kind='bar', ax=ax[0])
    ax[0].set_title('Raw Observation Row Counts & Missingness')
    ax[0].set_ylabel('Count')
    ax[0].tick_params(axis='x', rotation=0)
    
    # Percentage Missing
    pct_missing = (df_quality['missing_observations_count'] / df_quality['total_row_count']) * 100
    sns.barplot(x=pct_missing.index, y=pct_missing.values, ax=ax[1], palette='Reds_r')
    ax[1].set_title('Percentage of Missing Observations (%)')
    ax[1].set_ylabel('Missing %')
    
    plt.tight_layout()
    plt.show()
else:
    print('quality_summary.json not found. Run scripts/download_market_data.py first.')

--- 
## 2. Price Levels & Yield Trends

In [ ]:
prices_path = PROCESSED_DIR / 'cleaned_prices.csv'
df_prices = pd.read_csv(prices_path, parse_dates=['DATE'])
df_prices = df_prices.set_index('DATE')

fig, axes = plt.subplots(len(df_prices.columns), 1, figsize=(12, 4 * len(df_prices.columns)), sharex=True)
if len(df_prices.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, df_prices.columns):
    ax.plot(df_prices.index, df_prices[col], label=col, color='navy')
    ax.set_title(f'Cleaned Trajectory: {col}')
    ax.set_ylabel('Level / Percent')
    ax.legend(loc='upper left')

plt.xlabel('Date')
plt.tight_layout()
plt.show()

--- 
## 3. Return Distributions & Volatility Clusters

In [ ]:
returns_path = PROCESSED_DIR / 'market_returns.csv'
df_returns = pd.read_csv(returns_path, parse_dates=['DATE'])
df_returns = df_returns.set_index('DATE')

# Select log returns and yield changes for analysis
return_cols = [c for c in df_returns.columns if 'log_return' in c or 'daily_change_bps' in c]

# 3.1 Return Series Over Time
plt.figure(figsize=(12, 6))
for col in return_cols:
    plt.plot(df_returns.index, df_returns[col], alpha=0.7, label=col)
plt.title('Daily Log Returns & Yield Changes (Basis Points)')
plt.ylabel('Daily Delta')
plt.legend()
plt.show()

# 3.2 Distribution & Normality Check (QQ-Plot)
fig, axes = plt.subplots(len(return_cols), 2, figsize=(14, 4 * len(return_cols)))
if len(return_cols) == 1:
    axes = [axes]

for i, col in enumerate(return_cols):
    # Histogram / KDE
    sns.histplot(df_returns[col], kde=True, ax=axes[i][0], color='teal', stat='density')
    axes[i][0].set_title(f'Distribution: {col}')
    
    # Q-Q Plot
    stats.probplot(df_returns[col].dropna(), dist='norm', plot=axes[i][1])
    axes[i][1].set_title(f'Q-Q Plot vs Normal: {col}')

plt.tight_layout()
plt.show()

--- 
## 4. Asset Return Correlations

In [ ]:
plt.figure(figsize=(8, 6))
corr_matrix = df_returns[return_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f')
plt.title('Correlation Matrix (Log Returns & Yield Delta)')
plt.show()